# T5 · Exportar T-DEED a ONNX

Ejecuta las celdas de arriba abajo. **Cada una comprueba lo suyo y se para si algo falta**,
así que no se puede llegar al final con media cosa a medias.

Lo único que tienes que tocar es la ruta de los pesos, en la celda 3.

Al terminar te bajas dos ficheros:

- `tdeed-snb.onnx` → a `models/onnx/` del repo de detección
- `registry-block.yaml` → su contenido, dentro de `models:` en `models/registry.yaml`

Con GPU va más rápido, pero **no hace falta**: exportar es una pasada hacia
delante y en CPU sale el mismo grafo, solo que tarda unos minutos más.

## 1 · Entorno y código

In [ ]:
import pathlib
import subprocess
import sys


def corre(orden, cwd=None):
    """Lanza una orden y para aquí mismo si falla, enseñando el motivo."""
    salida = subprocess.run(orden, cwd=cwd, capture_output=True, text=True, check=False)
    if salida.returncode != 0:
        print(salida.stdout, salida.stderr)
        raise SystemExit(f"fallo: {' '.join(orden)}")
    return salida.stdout


# La GPU acelera, pero exportar no la necesita: trazar es una pasada hacia delante. Sin
# ella tarda unos minutos más y sale el mismo grafo.
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True, check=False)
print("GPU       :", gpu.stdout.strip() if gpu.returncode == 0 else
      "ninguna (va por CPU; si quieres acelerar: Entorno de ejecución -> GPU)")

# Los dos repos, recién clonados. Esto es lo que evita correr una copia vieja.
corre(["rm", "-rf", "/content/T-DEED", "/content/training"])
corre(["git", "clone", "-q", "--depth", "1", "https://github.com/arturxe2/T-DEED", "/content/T-DEED"])
corre(["git", "clone", "-q", "--depth", "1",
       "https://github.com/jhquihuiri7/fooball_ai_training", "/content/training"])
print("T-DEED    :", corre(["git", "-C", "/content/T-DEED", "rev-parse", "--short", "HEAD"]).strip())
print("training  :", corre(["git", "-C", "/content/training", "rev-parse", "--short", "HEAD"]).strip())

## 2 · Dependencias

Con `sys.executable -m pip`, no con `!pip`: así se instalan en el mismo Python que va a
exportar. Con `!pip` a veces no es el mismo, y ese es el clásico «lo instalé y sigue
diciendo que no está».

**No se instala su `requirements.txt`**: pinea `torch==2.3.1` y pelea con Colab.

In [ ]:
corre([sys.executable, "-m", "pip", "-q", "install",
       "timm", "tabulate", "wandb", "onnx", "onnxruntime", "onnxscript"])

# Comprobado importándolo, que es lo único que demuestra que está donde hace falta.
import onnx
import onnxruntime
import onnxscript
import timm
import torch

print("torch     :", torch.__version__, "· cuda", torch.cuda.is_available())
print("onnxscript:", onnxscript.__version__)
print("onnx      :", onnx.__version__, "· onnxruntime", onnxruntime.__version__)

## 3 · Los pesos

Lo único que tienes que ajustar. Elige la línea que te sirva y borra la otra.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

PESOS = pathlib.Path("/content/drive/MyDrive/tdeed/checkpoint_best.pt")   # <-- ajusta
# PESOS = pathlib.Path("/content/checkpoint_best.pt")                     # si ya está aquí

if not PESOS.is_file():
    raise SystemExit(f"no están los pesos en {PESOS}")
print(f"pesos     : {PESOS}  ({PESOS.stat().st_size / 1e6:.0f} MB)")

## 4 · Exportar y verificar

Aquí pasan las dos cosas: el export con formas estáticas y **T6**, que compara el mismo
clip por torch y por onnxruntime. Es donde se sabe si el gate-shift del backbone sobrevive
al export.

Tarda unos minutos y no imprime nada mientras tanto.

In [ ]:
salida = subprocess.run(
    [sys.executable, "/content/training/tools/export_onnx.py",
     "--tdeed", "/content/T-DEED",
     "--weights", str(PESOS),
     "--out", "/content/modelo"],
    cwd="/content/T-DEED",
    capture_output=True,
    text=True,
    env={"WANDB_MODE": "disabled", "PATH": "/usr/bin:/bin:/usr/local/bin", "HOME": "/root"},
    check=False,
)
print(salida.stdout)

if salida.returncode != 0:
    print(f"----- fallo (codigo {salida.returncode}) -----")

if salida.returncode < 0:
    # Un codigo negativo es una senal: el proceso no se quejo, lo mataron. En Colab eso
    # es casi siempre el vigilante de memoria, y no hay traza que leer porque el proceso
    # ni se entero. -9 es SIGKILL.
    print("El proceso fue MATADO, no fallo: se quedo sin memoria.")
    print("Trazar 100 frames a 448x796 no cabe en la RAM del runtime de CPU.")
    print("Solucion: Entorno de ejecucion -> Cambiar tipo de entorno -> GPU,")
    print("y repetir desde la primera celda.")
elif salida.returncode > 0:
    print(salida.stderr[-4000:])


## 5 · La ficha

Esto es lo que se pega en `models/registry.yaml` del repo de detección. Míralo antes de
copiarlo: el `sha256`, la forma de entrada y el orden de clases salen del `.onnx`
exportado, no de lo que nadie recuerde.

In [ ]:
ficha = pathlib.Path("/content/modelo/registry-block.yaml")
if ficha.is_file():
    print(ficha.read_text())
else:
    print("no hay ficha: el export no llegó a terminar")

## 6 · Bajarse los dos ficheros

El `.onnx` va a `models/onnx/` del repo de detección y la ficha, dentro de `models:` en
`models/registry.yaml`. Allí se comprueba el SHA-256 al cargarlo, así que si se corrompe
en el camino te lo dirá.

In [ ]:
from google.colab import files

onnx_path = pathlib.Path("/content/modelo/tdeed-snb.onnx")
if onnx_path.is_file():
    print(f"{onnx_path.name}: {onnx_path.stat().st_size / 1e6:.0f} MB")
    files.download(str(onnx_path))
    files.download(str(ficha))
else:
    print("no hay .onnx que bajar")

## Si algo falla

Lo que salga en «----- error -----» de la celda 4 es lo único que hace falta para
diagnosticar. Los fallos de entorno ya están cubiertos aquí; lo que quede será del modelo,
y el mensaje mencionará una operación concreta —`roll`, `Slice`, `ScatterND`— que es
información de verdad sobre si este camino sirve.